Install Dependencies

In [ ]:
!pip install -q bitsandbytes accelerate peft transformers datasets trl fsspec==2025.3.2 -U

Set Random Seeds for Reproducibility

In [ ]:
import random
import numpy as np
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Optional: make PyTorch even more deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Set dataset path

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Bot/data/training_data_multiturn.jsonl"
print("Dataset path:", DATASET_PATH)

import os

if os.path.exists(DATASET_PATH):
    print("✅ Dataset found!")
else:
    print("❌ Dataset not found. Check the path.")


Convert Dataset for Training

In [ ]:
import json

# Use the Google Drive project path for input and output files
input_file = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Bot/data/training_data_multiturn.jsonl"
output_file = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Bot/data/training_data_for_lora.jsonl"

with open(input_file, "r", encoding="utf-8") as f_in, open(output_file, "w", encoding="utf-8") as f_out:
    for line in f_in:
        data = json.loads(line)
        converted = {
            "instruction": data["prompt"].strip(),
            "output": data["response"].strip()
        }
        json.dump(converted, f_out, ensure_ascii=False)
        f_out.write("\n")

print(f"✅ Converted to {output_file}")


Load Converted Dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Use full paths for Drive-based project directory
converted_path = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Bot/data/training_data_for_lora.jsonl"
train_path = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Bot/data/train_split.jsonl"
val_path = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Bot/data/val_split.jsonl"

# Load the converted dataset as DataFrame
df = pd.read_json(converted_path, lines=True)

# Stratified split (or random if no source_user)
if 'source_user' in df.columns:
    stratify_col = df['source_user']
else:
    stratify_col = None

train_df, val_df = train_test_split(
    df,
    test_size=0.05,
    stratify=stratify_col,
    random_state=42
)

# Save the splits to JSONL files in your project directory
train_df.to_json(train_path, orient="records", lines=True)
val_df.to_json(val_path, orient="records", lines=True)

print(f"✅ Train/val split done. Train size: {len(train_df)}, Val size: {len(val_df)}")
print("Train split path:", train_path)
print("Val split path:", val_path)


Load Datasets Separately

In [ ]:
import pandas as pd
from datasets import Dataset

# Use your Drive paths
train_file = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Bot/data/train_split.jsonl"
val_file = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Bot/data/val_split.jsonl"

# Load using pandas
train_df = pd.read_json(train_file, lines=True)
val_df = pd.read_json(val_file, lines=True)

# Convert to HuggingFace Dataset objects
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

print(f"✅ Loaded train dataset with {len(train_dataset)} samples")
print(f"✅ Loaded val dataset with {len(val_dataset)} samples")


Log in to Hugging Face

In [ ]:
from huggingface_hub import login

login()


Load Model + Tokenizer (4-bit Quantized)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# Load base model + tokenizer
model_name = "mistralai/Mistral-7B-v0.1"

# Use bfloat16 if supported, else fallback to float16
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = dict(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", **bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Prepare for LoRA fine-tuning
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("✅ Model loaded, quantized, and LoRA-ready.")


Train Model with LoRA

In [ ]:
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer

# === Training Settings ===
PER_DEVICE_BATCH_SIZE = 24  # Adjust based on VRAM
NUM_EPOCHS = 2
EVAL_INTERVAL = 100  # Evaluate every 100 steps

# === Custom Eval Callback ===
class StepEvalCallback(TrainerCallback):
    def __init__(self, eval_interval=EVAL_INTERVAL):
        self.eval_interval = eval_interval

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.eval_interval == 0 and state.global_step > 0:
            print(f"\n Evaluating at step {state.global_step}")
            control.should_evaluate = True
        return control

# === Training Args ===
training_args = TrainingArguments(
    output_dir="gurt-mistral-lora",
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=1,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    bf16=True,
    dataloader_num_workers=2,
    report_to="none",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
)

# === Trainer ===
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="instruction",
    args=training_args,
    max_seq_length=2048,
    callbacks=[StepEvalCallback()],
)

# === Train ===
trainer.train()


Plot Training/Validation Loss After Training

In [ ]:
import matplotlib.pyplot as plt

# Extract training loss and steps
train_loss = [log['loss'] for log in trainer.state.log_history if 'loss' in log and 'eval_loss' not in log]
train_steps = [log['step'] for log in trainer.state.log_history if 'loss' in log and 'eval_loss' not in log]

# Extract validation loss and steps
eval_loss = [log['eval_loss'] for log in trainer.state.log_history if 'eval_loss' in log]
eval_steps = [log['step'] for log in trainer.state.log_history if 'eval_loss' in log]

# Plot
plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_loss, label='Train Loss')
plt.plot(eval_steps, eval_loss, label='Validation Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Steps')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


Save LoRA Adapter & Tokenizer + Export to Drive

In [ ]:
import os
import time

# Define where to save in Colab
save_dir = "/content/gurt-mistral-lora"

# Save LoRA adapter and tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ Model and tokenizer saved to: {save_dir}")

# List saved files for confirmation
print("Saved files:", os.listdir(save_dir))

# Ensure destination directory exists in Google Drive
drive_dir = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Bot"
if not os.path.exists(drive_dir):
    os.makedirs(drive_dir)

# Save to WhatsApp Bot project directory with timestamp
timestamp = time.strftime("%Y%m%d-%H%M%S")
backup_dir = f"{drive_dir}/gurt-mistral-lora-{timestamp}"

# Copy to Google Drive with timestamped directory
!cp -r {save_dir} "{backup_dir}"
print(f"✅ Backup saved to Google Drive at: {backup_dir}")


Generate Sample Completions After Training

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import pipeline

# Load model and tokenizer from your save_dir
model_path = save_dir
pipe = pipeline(
    "text-generation",
    model=model_path,
    tokenizer=model_path,
    device=0,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# === Reward Scoring Function ===
def reward_score(text):
    score = 0
    if "�" in text:
        return -1000
    if len(text.strip()) < 10:
        return -100
    if "Gurt:" not in text:
        score -= 5
    score += text.count("!") * 0.5
    score += sum(word in text.lower() for word in ["lol", "fr", "bro", "nah"]) * 1
    score -= sum(word in text.lower() for word in ["stfu", "fat", "ugly", "kill"]) * 2
    return score

# === Best-Response Generator ===
def generate_best_response(prompt, num_candidates=4):
    outputs = pipe(
        prompt,
        max_new_tokens=40,
        top_p=0.92,
        temperature=0.7,
        do_sample=True,
        num_return_sequences=num_candidates,
        clean_up_tokenization_spaces=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    candidates = [o["generated_text"] for o in outputs]
    best = sorted(candidates, key=reward_score, reverse=True)[0]
    return best

# === Test Prompt ===
prompt = "Mark: Bro did you actually clean the sink this time?\nElias:\nGurt:"
response = generate_best_response(prompt)

print("Prompt:\n", prompt)
print("Model completion:\n", response[len(prompt):])  # Show generated part only


Save config/hyperparameters as a text file

In [ ]:
config_info = f"""
Model: {model_name}
Batch size: {PER_DEVICE_BATCH_SIZE}
Learning rate: 2e-4
Epochs: 2
Scheduler: cosine
Warmup ratio: 0.05
Context window (max_seq_length): 2048
LoRA r: 16
LoRA alpha: 32
LoRA dropout: 0.05
Target modules: ['q_proj', 'v_proj']
"""

with open(f"{save_dir}/training_config.txt", "w") as f:
    f.write(config_info.strip())

print(f"✅ Training config saved to {save_dir}/training_config.txt")
